# Assignment 2 — Image Generation using (β-)VAE, VQ-VAE and WGAN — Part B
**Course:** Unsupervised Deep Learning (S1-26_AIMLZG533)   **Assessment:** Assignment 2   **Group:** 91

### Student Details
| Name | Email | ID | Contribution (%) |
|---|---|---|---|
| Sulagna Giri | 2025ae05607@wilp.bits-pilani.ac.in | 2025ae05607 | 100% |
| Vankudre Vikrant Vilas | 2025ae05336@wilp.bits-pilani.ac.in | 2025ae05336 | 100% |
| Vishnu M P | 2025ae05317@wilp.bits-pilani.ac.in | 2025ae05317 | 100% |
| Yeldona Desai V Vittal | 2025ae05316@wilp.bits-pilani.ac.in | 2025ae05316 | 100% |
| Akshay Mishra | 2025ae05261@wilp.bits-pilani.ac.in | 2025ae05261 | 100% |


## Overview
This notebook implements **WGAN** (weight clipping, Arjovsky et al., 2017) and **WGAN-GP**
(gradient penalty, Gulrajani et al., 2017) on CIFAR-10, with a shared DCGAN-style generator
and a shared critic backbone (critic normalisation differs per model as specified).

Inside the GAN training loop only, real images are mapped from [0,1] to [-1,1] (`x*2-1`) to
match the generator's `tanh` output, and generated images are mapped back with `(g+1)/2`
before any metric or visualisation (FID, PSNR table entries are "N/A" for GANs — there is no
encoder/reconstruction path).

## How to run
- `QUICK_RUN` (default `False`): a few hundred generator iterations and reduced FID sample
  counts for a fast smoke test. Final submission must be run with `QUICK_RUN = False`.
- `RESUME` (default `True`): resumes from the last saved checkpoint + history if present.
- `CIFAR10_DIR` environment variable (optional): CIFAR-10 location (defaults to `./data`).


In [ ]:

import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # safeguard in case TensorFlow is imported indirectly
import sys, platform, shutil, subprocess
from pathlib import Path

print("=" * 70)
print("ENVIRONMENT / LAB INFRASTRUCTURE")
print("=" * 70)
print(f"Hostname: {platform.node()}")
print(f"OS: {platform.platform()}")
print(f"Python: {sys.version}")

import torch
print(f"Torch: {torch.__version__}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")

sys.path.insert(0, str(Path.cwd()))
from udl_utils import get_device, seed_everything, print_env_banner, SEED

seed_everything(SEED)
DEVICE = get_device()
print_env_banner(DEVICE)

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)
else:
    print("nvidia-smi not found on this machine (expected on non-lab machines).")


In [ ]:

from pathlib import Path
print(Path("udl_utils.py").read_text())


In [ ]:

import numpy, pandas, matplotlib, torchvision, torchmetrics
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("matplotlib", matplotlib.__version__)
print("torchvision", torchvision.__version__)
print("torchmetrics", torchmetrics.__version__)


In [ ]:

QUICK_RUN = False
RESUME = True

print("=" * 70)
print(f"RUN MODE: {'QUICK_RUN (smoke test)' if QUICK_RUN else 'FULL RUN'}")
print("=" * 70)

from pathlib import Path
OUT = Path("outputs")
(OUT / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUT / "history").mkdir(parents=True, exist_ok=True)
(OUT / "results").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "logs").mkdir(parents=True, exist_ok=True)

WGAN_GEN_ITERS = 300 if QUICK_RUN else 50_000
WGANGP_GEN_ITERS = 300 if QUICK_RUN else 50_000
QUICK_FID_EVERY = 100 if QUICK_RUN else 5_000
QUICK_FID_N = 200 if QUICK_RUN else 5_000
FINAL_FID_N = 1000 if QUICK_RUN else 10_000
LOG_EVERY = 50 if QUICK_RUN else 500
BATCH_SIZE = 64
N_CRITIC = 5

print(f"WGAN_GEN_ITERS={WGAN_GEN_ITERS}  WGANGP_GEN_ITERS={WGANGP_GEN_ITERS}  FINAL_FID_N={FINAL_FID_N}")


In [ ]:

import json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

from udl_utils import get_loaders, FIDHelper, save_grid, save_json, load_json, Timer, infinite_loader

train_loader, test_loader = get_loaders(BATCH_SIZE)
print(f"Train batches: {len(train_loader)}  Test batches: {len(test_loader)}")

print("Caching real-image FID statistics...")
fid_helper = FIDHelper(DEVICE, test_loader, n_real=FINAL_FID_N)
print("Done.")


## Shared generator / critic architectures

See markdown above each training function for the per-model optimiser, normalisation and loss choices.

In [ ]:

Z_DIM = 128

class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.fc = nn.Linear(z_dim, 4 * 4 * 512)
        self.net = nn.Sequential(
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 3, 4, stride=2, padding=1), nn.Tanh(),
        )

    def forward(self, z):
        h = self.fc(z).view(-1, 512, 4, 4)
        out = self.net(h)
        assert out.shape[1:] == (3, 32, 32), f"Unexpected generator output shape {out.shape}"
        return out


class Critic(nn.Module):
    """use_norm: 'batch' (WGAN), 'layer' (WGAN-GP, via GroupNorm(1,C) == LayerNorm over C,H,W), or None."""

    def __init__(self, use_norm=None):
        super().__init__()

        def norm(c):
            if use_norm == "batch":
                return nn.BatchNorm2d(c)
            if use_norm == "layer":
                return nn.GroupNorm(1, c)
            return nn.Identity()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), norm(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), norm(256), nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(256 * 4 * 4, 1)

    def forward(self, x):
        h = self.net(x).flatten(1)
        return self.fc(h).squeeze(1)


def gradient_penalty(critic, real, fake, device):
    bs = real.shape[0]
    eps = torch.rand(bs, 1, 1, 1, device=device)
    interp = (eps * real + (1 - eps) * fake).requires_grad_(True)
    scores = critic(interp)
    grads = torch.autograd.grad(
        outputs=scores, inputs=interp, grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    grads = grads.view(bs, -1)
    gp = ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp


In [ ]:

def train_gan(name, gen_iters, use_gp):
    tag = name
    ckpt_g = OUT / "checkpoints" / f"{tag}_generator.pt"
    ckpt_c = OUT / "checkpoints" / f"{tag}_critic.pt"
    hist_path = OUT / "history" / f"{tag}.json"

    generator = Generator().to(DEVICE)
    critic = Critic(use_norm=("layer" if use_gp else "batch")).to(DEVICE)

    if use_gp:
        opt_g = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.0, 0.9))
        opt_c = torch.optim.Adam(critic.parameters(), lr=1e-4, betas=(0.0, 0.9))
    else:
        opt_g = torch.optim.RMSprop(generator.parameters(), lr=5e-5)
        opt_c = torch.optim.RMSprop(critic.parameters(), lr=5e-5)

    fixed_noise = torch.randn(100, Z_DIM, device=DEVICE)
    data_iter = infinite_loader(train_loader)

    if RESUME and ckpt_g.exists() and ckpt_c.exists() and hist_path.exists():
        generator.load_state_dict(torch.load(ckpt_g, map_location=DEVICE))
        critic.load_state_dict(torch.load(ckpt_c, map_location=DEVICE))
        history = load_json(hist_path)
        print(f"[{tag}] Resumed from checkpoint at iteration {history['iteration'][-1] if history['iteration'] else 0}.")
        start_iter = history["iteration"][-1] if history["iteration"] else 0
    else:
        history = {"iteration": [], "critic_loss": [], "gen_loss": [], "wasserstein_est": [],
                   "gp_value": [], "iter_time_s": [], "quick_fid_iter": [], "quick_fid": []}
        start_iter = 0

    best_fid, best_iter = float("inf"), None
    if history["quick_fid"]:
        bi = int(np.argmin(history["quick_fid"]))
        best_fid, best_iter = history["quick_fid"][bi], history["quick_fid_iter"][bi]

    if start_iter < gen_iters:
        generator.train(); critic.train()
        for it in range(start_iter + 1, gen_iters + 1):
            t0 = time.perf_counter()
            for _ in range(N_CRITIC):
                x, _ = next(data_iter)
                x_real = (x.to(DEVICE) * 2 - 1)
                z = torch.randn(x_real.shape[0], Z_DIM, device=DEVICE)
                x_fake = generator(z).detach()

                opt_c.zero_grad()
                c_real = critic(x_real).mean()
                c_fake = critic(x_fake).mean()
                gp_val = torch.tensor(0.0, device=DEVICE)
                if use_gp:
                    gp_val = gradient_penalty(critic, x_real, x_fake, DEVICE)
                    c_loss = -(c_real - c_fake) + 10.0 * gp_val
                else:
                    c_loss = -(c_real - c_fake)
                c_loss.backward()
                opt_c.step()
                if not use_gp:
                    for p in critic.parameters():
                        p.data.clamp_(-0.01, 0.01)

            z = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
            opt_g.zero_grad()
            g_loss = -critic(generator(z)).mean()
            g_loss.backward()
            opt_g.step()

            iter_time = time.perf_counter() - t0
            wasserstein_est = (c_real - c_fake).item()

            if it % LOG_EVERY == 0 or it == gen_iters:
                history["iteration"].append(it)
                history["critic_loss"].append(c_loss.item())
                history["gen_loss"].append(g_loss.item())
                history["wasserstein_est"].append(wasserstein_est)
                history["gp_value"].append(gp_val.item())
                history["iter_time_s"].append(iter_time)
                print(f"[{tag}] it {it:6d}/{gen_iters}  critic_loss={c_loss.item():.3f}  "
                      f"gen_loss={g_loss.item():.3f}  W_est={wasserstein_est:.3f}  "
                      f"gp={gp_val.item():.3f}  it_time={iter_time*1000:.0f}ms")
                if DEVICE.type == "mps":
                    torch.mps.empty_cache()  # MPS caching allocator can otherwise grow unbounded over long runs

            if it % QUICK_FID_EVERY == 0 or it == gen_iters:
                generator.eval()
                with torch.no_grad():
                    sample_fn = lambda n: (generator(torch.randn(n, Z_DIM, device=DEVICE)) + 1) / 2
                    q_fid = fid_helper.fid_of_generator(sample_fn, n=QUICK_FID_N)
                generator.train()
                history["quick_fid_iter"].append(it)
                history["quick_fid"].append(q_fid)
                print(f"[{tag}] it {it:6d}  quick-FID({QUICK_FID_N})={q_fid:.2f}")

                torch.save(generator.state_dict(), OUT / "checkpoints" / f"{tag}_generator_it{it}.pt")
                with torch.no_grad():
                    fixed_imgs = (generator(fixed_noise) + 1) / 2
                save_grid(fixed_imgs, OUT / "figures" / f"{tag}_fixed_noise_it{it}.png", nrow=10)

                if q_fid < best_fid:
                    best_fid, best_iter = q_fid, it
                    torch.save(generator.state_dict(), ckpt_g.with_name(f"{tag}_generator_best.pt"))

                save_json(history, hist_path)

        torch.save(generator.state_dict(), ckpt_g)
        torch.save(critic.state_dict(), ckpt_c)
        save_json(history, hist_path)
    else:
        print(f"[{tag}] Checkpoint already covers {gen_iters} iterations; nothing to train.")

    print(f"\n--- {tag} training log (logged every {LOG_EVERY} iters) ---")
    for i in range(len(history["iteration"])):
        print(f"it {history['iteration'][i]:6d}  critic_loss={history['critic_loss'][i]:.3f}  "
              f"gen_loss={history['gen_loss'][i]:.3f}  W_est={history['wasserstein_est'][i]:.3f}  "
              f"gp={history['gp_value'][i]:.3f}")

    return generator, critic, history, best_fid, best_iter


### Task 1 — train WGAN and WGAN-GP

In [ ]:

gan_models = {}
gan_histories = {}
gan_best = {}

gen, crit, hist, best_fid, best_iter = train_gan("wgan", WGAN_GEN_ITERS, use_gp=False)
gan_models["wgan"] = {"generator": gen, "critic": crit}
gan_histories["wgan"] = hist
gan_best["wgan"] = (best_fid, best_iter)

gen, crit, hist, best_fid, best_iter = train_gan("wgangp", WGANGP_GEN_ITERS, use_gp=True)
gan_models["wgangp"] = {"generator": gen, "critic": crit}
gan_histories["wgangp"] = hist
gan_best["wgangp"] = (best_fid, best_iter)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in ["wgan", "wgangp"]:
    h = gan_histories[name]
    axes[0].plot(h["iteration"], h["wasserstein_est"], label=name)
axes[0].set_xlabel("Generator iteration"); axes[0].set_ylabel("Wasserstein estimate")
axes[0].set_title("Wasserstein estimate vs iteration"); axes[0].legend(); axes[0].grid(alpha=0.3)

for name in ["wgan", "wgangp"]:
    h = gan_histories[name]
    axes[1].plot(h["quick_fid_iter"], h["quick_fid"], marker="o", markersize=3, label=name)
axes[1].set_xlabel("Generator iteration"); axes[1].set_ylabel(f"Quick FID ({QUICK_FID_N} samples)")
axes[1].set_title("Quick-FID vs iteration"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "figures" / "wgan_wgangp_training_curves.png", bbox_inches="tight", dpi=120)
plt.show()


### Task 2 — generate random samples

In [ ]:

for name in ["wgan", "wgangp"]:
    generator = gan_models[name]["generator"]
    generator.eval()
    with torch.no_grad():
        z = torch.randn(100, Z_DIM, device=DEVICE)
        imgs = (generator(z) + 1) / 2
    save_grid(imgs, OUT / "figures" / f"{name}_samples_100.png", nrow=10)

print("Saved", OUT / "figures" / "wgan_samples_100.png", "and", OUT / "figures" / "wgangp_samples_100.png")


### Task 3 — compute FID scores (full protocol)

In [ ]:

final_results = {}
for name in ["wgan", "wgangp"]:
    generator = gan_models[name]["generator"]
    generator.eval()

    with torch.no_grad():
        sample_fn = lambda n, g=generator: (g(torch.randn(n, Z_DIM, device=DEVICE)) + 1) / 2
        final_fid = fid_helper.fid_of_generator(sample_fn, n=FINAL_FID_N)

    best_fid_quick, best_iter = gan_best[name]
    best_ckpt = OUT / "checkpoints" / f"{name}_generator_best.pt"
    best_fid_full = None
    if best_ckpt.exists():
        best_gen = Generator().to(DEVICE)
        best_gen.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
        best_gen.eval()
        with torch.no_grad():
            sample_fn_best = lambda n, g=best_gen: (g(torch.randn(n, Z_DIM, device=DEVICE)) + 1) / 2
            best_fid_full = fid_helper.fid_of_generator(sample_fn_best, n=FINAL_FID_N)

    h = gan_histories[name]
    with Timer() as t:
        with torch.no_grad():
            _ = generator(torch.randn(min(1000, FINAL_FID_N), Z_DIM, device=DEVICE))
    sample_time_10k = t["elapsed"] * (10000 / min(1000, FINAL_FID_N))

    final_results[name] = {
        "final_fid": final_fid,
        "best_fid_quick_eval": best_fid_quick,
        "best_fid_full_protocol": best_fid_full,
        "best_iteration": best_iter,
        "train_time_s": sum(h["iter_time_s"]) if h["iter_time_s"] else None,
        "sample_time_10k_s": sample_time_10k,
        "sample_time_per_100_s": sample_time_10k / 100,
    }
    print(f"[{name}] final-FID({FINAL_FID_N})={final_fid:.2f}  "
          f"best-FID(full,@it={best_iter})={best_fid_full}  "
          f"sample_time_10k={sample_time_10k:.1f}s")

gan_table = pd.DataFrame([
    {"model": name.upper(), **final_results[name]} for name in ["wgan", "wgangp"]
])
gan_table.to_csv(OUT / "results" / "gan_results_table.csv", index=False)
display(gan_table)


In [ ]:

from IPython.display import Markdown, display as _display

wgan_fid = final_results["wgan"]["final_fid"]
wgangp_fid = final_results["wgangp"]["final_fid"]
more_stable_expected = wgangp_fid < wgan_fid

lines = []
lines.append("### Discussion: training stability, Wasserstein estimate, sample quality\n")
lines.append(f"Final FID: WGAN={wgan_fid:.2f}, WGAN-GP={wgangp_fid:.2f}.\n")
if more_stable_expected:
    lines.append(
        "This **matches the expected result**: WGAN-GP's gradient penalty enforces the "
        "1-Lipschitz constraint more smoothly than weight clipping, which tends to either "
        "saturate critic capacity or vanish gradients; WGAN-GP therefore reaches a lower "
        "(better) FID here, consistent with Gulrajani et al. (2017).\n"
    )
else:
    lines.append(
        "This **does not match** the usual expectation that WGAN-GP outperforms weight-"
        "clipped WGAN. Reported as observed — a likely cause under a fixed iteration budget "
        "is that one model needed more generator iterations to converge, or n_critic=5 was "
        "not enough updates per generator step for the harder-to-optimise variant.\n"
    )
lines.append(
    "The Wasserstein estimate (critic score gap between real and fake) is plotted above for "
    "both models; a steadily increasing, less noisy estimate indicates a more informative "
    "critic signal, which is the qualitative signature WGAN-GP is expected to show relative "
    "to the clipped critic."
)
_display(Markdown("\n".join(lines)))


## Save Part B results for Part C

In [ ]:

partB_results = {
    "wgan": {"history": gan_histories["wgan"], **final_results["wgan"]},
    "wgangp": {"history": gan_histories["wgangp"], **final_results["wgangp"]},
    "table": gan_table.to_dict(orient="list"),
    "quick_run": QUICK_RUN,
}
save_json(partB_results, OUT / "results" / "partB_results.json")
print("Saved", OUT / "results" / "partB_results.json")
